# Prompt Template A/B Testing and Evaluation

## Learning Objectives

By completing this notebook, you will:
1. Design and execute A/B tests for prompt variants
2. Implement quantitative evaluation metrics for LLM outputs
3. Use statistical methods to compare prompt performance
4. Optimize prompts based on evaluation results
5. Create automated test suites for prompt quality assurance

## Prerequisites

- Module 1: Notebook 01 completed (Prompt creation basics)
- Understanding of basic statistics (mean, variance, significance testing)
- Familiarity with JSON and structured data

## Estimated Time: 60 minutes

---

## Why Prompt Testing Matters

**The Challenge**: How do you know if your prompt is actually good?

In traditional software, we write unit tests. For LLMs, we need:
- **Quantitative metrics** (accuracy, format compliance, cost)
- **Qualitative assessment** (relevance, tone, completeness)
- **A/B comparisons** (which prompt variant performs better?)
- **Regression testing** (did changes break existing functionality?)

### Key Insight

**Small prompt changes can have large effects**. Testing isn't optional - it's essential for production reliability.

### Dataiku LLM Evaluation

Dataiku provides tools for:
1. **Test dataset management** - Store ground truth examples
2. **Automated evaluation** - Run prompts against test sets
3. **Metric calculation** - Accuracy, latency, cost tracking
4. **Visual comparison** - Side-by-side prompt performance

## Setup

Import libraries for prompt testing and evaluation.

In [ ]:
# Purpose: Setup testing framework
# Key Concept: Systematic prompt evaluation requires structured testing

import dataiku
from dataiku.llm import LLM
import json
import time
from typing import Dict, List, Callable, Tuple
from dataclasses import dataclass
from collections import defaultdict
import statistics
import re

# For statistical testing
from scipy import stats
import numpy as np

# For visualization
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("✓ Testing framework loaded")

## Building a Test Framework

A robust prompt testing framework needs:
1. **Test cases** - Input + expected output
2. **Evaluation metrics** - How to score outputs
3. **Comparison logic** - Statistical significance testing
4. **Result storage** - Track performance over time

In [ ]:
@dataclass
class TestCase:
    """Represents a single test case for prompt evaluation."""
    id: str
    input: Dict[str, str]
    expected_output: Dict[str, any]
    category: str = "general"

@dataclass
class EvaluationResult:
    """Results from evaluating a prompt on a test case."""
    test_id: str
    prompt_version: str
    actual_output: str
    scores: Dict[str, float]
    latency_ms: float
    tokens_used: int
    passed: bool

class PromptEvaluator:
    """
    Framework for systematic prompt testing.
    
    Supports:
    - Multiple evaluation metrics
    - A/B testing between prompt variants
    - Statistical significance testing
    - Cost and latency tracking
    """
    
    def __init__(self, llm: LLM):
        self.llm = llm
        self.test_cases: List[TestCase] = []
        self.results: List[EvaluationResult] = []
        self.metrics: Dict[str, Callable] = {}
    
    def add_test_case(self, test_case: TestCase):
        """Add a test case to the suite."""
        self.test_cases.append(test_case)
    
    def add_metric(self, name: str, metric_fn: Callable[[str, Dict], float]):
        """
        Add an evaluation metric.
        
        Parameters
        ----------
        name : str
            Metric name (e.g., 'accuracy', 'format_compliance')
        metric_fn : Callable
            Function that takes (actual_output, expected_output) and returns score 0-1
        """
        self.metrics[name] = metric_fn
    
    def evaluate_prompt(self, prompt_template: str, version_name: str) -> Dict:
        """
        Evaluate a prompt template on all test cases.
        
        Returns
        -------
        dict
            Aggregated metrics, per-test results, timing info
        """
        version_results = []
        
        for test_case in self.test_cases:
            # Render prompt with test inputs
            prompt = prompt_template.format(**test_case.input)
            
            # Execute and time
            start = time.time()
            response = self.llm.complete(prompt, max_tokens=300, temperature=0)
            latency = (time.time() - start) * 1000  # ms
            
            # Calculate all metrics
            scores = {}
            for metric_name, metric_fn in self.metrics.items():
                scores[metric_name] = metric_fn(response.text, test_case.expected_output)
            
            # Overall pass/fail
            passed = all(score >= 0.7 for score in scores.values())
            
            result = EvaluationResult(
                test_id=test_case.id,
                prompt_version=version_name,
                actual_output=response.text,
                scores=scores,
                latency_ms=latency,
                tokens_used=response.usage.total_tokens,
                passed=passed
            )
            
            version_results.append(result)
            self.results.append(result)
        
        # Aggregate metrics
        return self._aggregate_results(version_results, version_name)
    
    def _aggregate_results(self, results: List[EvaluationResult], version: str) -> Dict:
        """Calculate aggregate statistics for a prompt version."""
        if not results:
            return {}
        
        # Aggregate each metric
        metric_scores = defaultdict(list)
        for result in results:
            for metric, score in result.scores.items():
                metric_scores[metric].append(score)
        
        aggregated = {
            'version': version,
            'total_tests': len(results),
            'passed_tests': sum(r.passed for r in results),
            'pass_rate': sum(r.passed for r in results) / len(results),
            'avg_latency_ms': statistics.mean(r.latency_ms for r in results),
            'total_tokens': sum(r.tokens_used for r in results),
            'avg_tokens': statistics.mean(r.tokens_used for r in results),
            'metrics': {}
        }
        
        for metric, scores in metric_scores.items():
            aggregated['metrics'][metric] = {
                'mean': statistics.mean(scores),
                'median': statistics.median(scores),
                'stdev': statistics.stdev(scores) if len(scores) > 1 else 0,
                'min': min(scores),
                'max': max(scores)
            }
        
        return aggregated
    
    def compare_versions(self, version_a: str, version_b: str) -> Dict:
        """
        Statistical comparison of two prompt versions.
        
        Uses t-test to determine if performance difference is significant.
        """
        results_a = [r for r in self.results if r.prompt_version == version_a]
        results_b = [r for r in self.results if r.prompt_version == version_b]
        
        if not results_a or not results_b:
            return {'error': 'Missing results for one or both versions'}
        
        comparison = {'version_a': version_a, 'version_b': version_b, 'metrics': {}}
        
        # Compare each metric
        metric_names = set(results_a[0].scores.keys())
        
        for metric in metric_names:
            scores_a = [r.scores[metric] for r in results_a]
            scores_b = [r.scores[metric] for r in results_b]
            
            # Perform t-test
            t_stat, p_value = stats.ttest_ind(scores_a, scores_b)
            
            # Calculate effect size (Cohen's d)
            pooled_std = np.sqrt((np.var(scores_a) + np.var(scores_b)) / 2)
            cohens_d = (np.mean(scores_a) - np.mean(scores_b)) / pooled_std if pooled_std > 0 else 0
            
            comparison['metrics'][metric] = {
                'mean_a': np.mean(scores_a),
                'mean_b': np.mean(scores_b),
                'difference': np.mean(scores_a) - np.mean(scores_b),
                'p_value': p_value,
                'significant': p_value < 0.05,
                'cohens_d': cohens_d,
                'winner': version_a if np.mean(scores_a) > np.mean(scores_b) else version_b
            }
        
        return comparison

print("✓ PromptEvaluator framework defined")

## Defining Evaluation Metrics

Good metrics are:
- **Objective** - Same result every time
- **Relevant** - Measure what matters for your use case
- **Actionable** - Poor scores suggest how to improve

Common metric types:
1. **Format compliance** - Is output valid JSON? Correct fields?
2. **Exact match** - Does output exactly match expected?
3. **Semantic similarity** - Is meaning preserved even if wording differs?
4. **Field accuracy** - Are individual fields correct?

In [ ]:
# Purpose: Define reusable evaluation metrics
# Key Concept: Different use cases need different metrics

def json_format_metric(actual: str, expected: Dict) -> float:
    """
    Check if output is valid JSON with expected keys.
    
    Returns 1.0 if valid JSON with all expected keys, 0.0 otherwise.
    """
    try:
        parsed = json.loads(actual)
        expected_keys = set(expected.keys())
        actual_keys = set(parsed.keys())
        
        # All expected keys present?
        if expected_keys.issubset(actual_keys):
            return 1.0
        else:
            # Partial credit for having some keys
            return len(expected_keys & actual_keys) / len(expected_keys)
    except json.JSONDecodeError:
        return 0.0

def field_accuracy_metric(actual: str, expected: Dict) -> float:
    """
    Check if individual fields match expected values.
    
    Returns proportion of fields that match.
    """
    try:
        parsed = json.loads(actual)
        
        matches = 0
        total = 0
        
        for key, expected_value in expected.items():
            if key in parsed:
                total += 1
                actual_value = parsed[key]
                
                # Handle different value types
                if isinstance(expected_value, str):
                    if str(actual_value).lower() == expected_value.lower():
                        matches += 1
                elif isinstance(expected_value, (int, float)):
                    # Allow 5% tolerance for numbers
                    if abs(float(actual_value) - expected_value) / expected_value < 0.05:
                        matches += 1
                else:
                    if actual_value == expected_value:
                        matches += 1
        
        return matches / total if total > 0 else 0.0
    except (json.JSONDecodeError, ValueError, ZeroDivisionError):
        return 0.0

def sentiment_accuracy_metric(actual: str, expected: Dict) -> float:
    """
    Specialized metric for sentiment classification.
    
    Checks if sentiment field matches expected value.
    """
    try:
        parsed = json.loads(actual)
        
        if 'sentiment' not in parsed or 'sentiment' not in expected:
            return 0.0
        
        actual_sentiment = parsed['sentiment'].lower().strip()
        expected_sentiment = expected['sentiment'].lower().strip()
        
        return 1.0 if actual_sentiment == expected_sentiment else 0.0
    except (json.JSONDecodeError, KeyError):
        return 0.0

def completeness_metric(actual: str, expected: Dict) -> float:
    """
    Check if all expected fields have non-null values.
    """
    try:
        parsed = json.loads(actual)
        
        complete_fields = 0
        for key in expected.keys():
            if key in parsed and parsed[key] is not None and parsed[key] != "":
                complete_fields += 1
        
        return complete_fields / len(expected) if expected else 0.0
    except json.JSONDecodeError:
        return 0.0

print("✓ Evaluation metrics defined")

## Exercise 1: Build a Test Suite

**Task**: Create a comprehensive test suite for sentiment analysis prompts.

Your test suite should include:
- At least 6 test cases
- Mix of bullish, bearish, and neutral sentiments
- Edge cases (ambiguous sentiment, missing info)
- Multiple commodity types

In [ ]:
# YOUR CODE HERE
# Create test cases for sentiment analysis

# Initialize evaluator
llm = LLM('claude-sonnet')  # Update with your connection
evaluator = PromptEvaluator(llm)

# Add metrics
evaluator.add_metric('json_format', json_format_metric)
evaluator.add_metric('sentiment_accuracy', sentiment_accuracy_metric)
evaluator.add_metric('completeness', completeness_metric)

# Create test cases
test_cases = [
    TestCase(
        id="test_001",
        input={"text": "Crude oil inventories plunged 7.5 million barrels, far exceeding analyst expectations."},
        expected_output={"sentiment": "bullish", "confidence": "high"},
        category="clear_bullish"
    ),
    # YOUR CODE HERE - Add 5 more test cases
    # Include:
    # - Clear bearish case
    # - Neutral case
    # - Ambiguous case
    # - Different commodity
    # - Complex multi-factor case
]

# Add test cases to evaluator
for test_case in test_cases:
    evaluator.add_test_case(test_case)

print(f"✓ Test suite created with {len(test_cases)} test cases")

# Auto-graded checks
assert len(test_cases) >= 6, "Test suite should have at least 6 test cases"
sentiments = [tc.expected_output['sentiment'] for tc in test_cases]
assert 'bullish' in sentiments, "Should include bullish test case"
assert 'bearish' in sentiments, "Should include bearish test case"
assert 'neutral' in sentiments, "Should include neutral test case"

print("✓ Exercise 1 passed!")

## Exercise 2: A/B Test Prompt Variants

Now let's test two different prompt approaches:

**Version A**: Simple, direct prompt
**Version B**: Detailed prompt with examples and context

In [ ]:
# Define prompt variants

prompt_v1_simple = """Analyze the sentiment of this commodity market report.

Report: {text}

Respond in JSON format:
{{
  "sentiment": "bullish" or "bearish" or "neutral",
  "confidence": "high" or "medium" or "low",
  "reason": "brief explanation"
}}

JSON:"""

# YOUR CODE HERE
# Create an improved version (prompt_v2_improved)
# Should include:
# - Better role definition
# - Few-shot examples
# - Clearer output format specification
# - Handling of edge cases

prompt_v2_improved = """YOUR IMPROVED PROMPT HERE
"""

# Evaluate both versions
print("Evaluating Version 1 (Simple)...")
results_v1 = evaluator.evaluate_prompt(prompt_v1_simple, "v1_simple")

print("\nEvaluating Version 2 (Improved)...")
results_v2 = evaluator.evaluate_prompt(prompt_v2_improved, "v2_improved")

# Display results
print("\n" + "="*60)
print("EVALUATION RESULTS")
print("="*60)

for results in [results_v1, results_v2]:
    print(f"\n{results['version'].upper()}:")
    print(f"  Pass Rate: {results['pass_rate']:.1%}")
    print(f"  Avg Latency: {results['avg_latency_ms']:.0f}ms")
    print(f"  Avg Tokens: {results['avg_tokens']:.0f}")
    print(f"  Metrics:")
    for metric_name, metric_data in results['metrics'].items():
        print(f"    {metric_name}: {metric_data['mean']:.2f} (±{metric_data['stdev']:.2f})")

# Statistical comparison
print("\n" + "="*60)
print("STATISTICAL COMPARISON")
print("="*60)

comparison = evaluator.compare_versions("v1_simple", "v2_improved")

for metric, stats in comparison['metrics'].items():
    print(f"\n{metric}:")
    print(f"  v1_simple: {stats['mean_a']:.3f}")
    print(f"  v2_improved: {stats['mean_b']:.3f}")
    print(f"  Difference: {stats['difference']:.3f}")
    print(f"  Significant: {'YES' if stats['significant'] else 'NO'} (p={stats['p_value']:.4f})")
    print(f"  Winner: {stats['winner']}")

# Auto-graded checks
assert results_v2['pass_rate'] >= results_v1['pass_rate'], "v2 should perform at least as well as v1"
print("\n✓ Exercise 2 passed!")

## Visualizing Test Results

Visual comparison makes it easier to spot performance differences.

In [ ]:
# Purpose: Visualize A/B test results
# Key Concept: Visualization reveals patterns metrics alone might miss

def plot_comparison(evaluator: PromptEvaluator, version_a: str, version_b: str):
    """Create comparison visualizations for two prompt versions."""
    
    # Get results for each version
    results_a = [r for r in evaluator.results if r.prompt_version == version_a]
    results_b = [r for r in evaluator.results if r.prompt_version == version_b]
    
    # Extract metric names
    metric_names = list(results_a[0].scores.keys())
    
    # Create subplots
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'Prompt Comparison: {version_a} vs {version_b}', fontsize=16, fontweight='bold')
    
    # 1. Metric comparison (bar chart)
    ax1 = axes[0, 0]
    metric_means_a = [np.mean([r.scores[m] for r in results_a]) for m in metric_names]
    metric_means_b = [np.mean([r.scores[m] for r in results_b]) for m in metric_names]
    
    x = np.arange(len(metric_names))
    width = 0.35
    
    ax1.bar(x - width/2, metric_means_a, width, label=version_a, alpha=0.8)
    ax1.bar(x + width/2, metric_means_b, width, label=version_b, alpha=0.8)
    ax1.set_ylabel('Score', fontsize=11)
    ax1.set_title('Metric Scores Comparison', fontsize=12, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(metric_names, rotation=45, ha='right')
    ax1.legend()
    ax1.set_ylim(0, 1.1)
    ax1.grid(axis='y', alpha=0.3)
    
    # 2. Latency comparison (box plot)
    ax2 = axes[0, 1]
    latency_data = [
        [r.latency_ms for r in results_a],
        [r.latency_ms for r in results_b]
    ]
    ax2.boxplot(latency_data, labels=[version_a, version_b])
    ax2.set_ylabel('Latency (ms)', fontsize=11)
    ax2.set_title('Response Latency Distribution', fontsize=12, fontweight='bold')
    ax2.grid(axis='y', alpha=0.3)
    
    # 3. Token usage
    ax3 = axes[1, 0]
    tokens_a = [r.tokens_used for r in results_a]
    tokens_b = [r.tokens_used for r in results_b]
    ax3.bar([version_a, version_b], 
            [np.mean(tokens_a), np.mean(tokens_b)],
            alpha=0.8,
            color=['#1f77b4', '#ff7f0e'])
    ax3.set_ylabel('Tokens', fontsize=11)
    ax3.set_title('Average Token Usage', fontsize=12, fontweight='bold')
    ax3.grid(axis='y', alpha=0.3)
    
    # 4. Pass/Fail rate
    ax4 = axes[1, 1]
    pass_rate_a = sum(r.passed for r in results_a) / len(results_a)
    pass_rate_b = sum(r.passed for r in results_b) / len(results_b)
    
    versions = [version_a, version_b]
    pass_rates = [pass_rate_a, pass_rate_b]
    fail_rates = [1 - pass_rate_a, 1 - pass_rate_b]
    
    ax4.bar(versions, pass_rates, label='Passed', alpha=0.8, color='green')
    ax4.bar(versions, fail_rates, bottom=pass_rates, label='Failed', alpha=0.8, color='red')
    ax4.set_ylabel('Proportion', fontsize=11)
    ax4.set_title('Test Pass/Fail Rate', fontsize=12, fontweight='bold')
    ax4.set_ylim(0, 1)
    ax4.legend()
    ax4.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Generate comparison visualization
plot_comparison(evaluator, "v1_simple", "v2_improved")

## Exercise 3: Optimize Based on Results

**Task**: Create a third version that addresses weaknesses found in testing.

Look at which test cases failed and why. Common issues:
- Format problems (not valid JSON)
- Missing required fields
- Incorrect sentiment classification
- Poor handling of edge cases

In [ ]:
# Analyze failures
print("Failed Test Cases Analysis")
print("="*60)

for result in evaluator.results:
    if not result.passed and result.prompt_version == "v2_improved":
        test_case = next(tc for tc in evaluator.test_cases if tc.id == result.test_id)
        print(f"\nTest ID: {result.test_id}")
        print(f"Input: {test_case.input['text'][:80]}...")
        print(f"Scores: {result.scores}")
        print(f"Output: {result.actual_output[:100]}...")

# YOUR CODE HERE
# Create v3_optimized that fixes the identified issues

prompt_v3_optimized = """YOUR OPTIMIZED PROMPT HERE
"""

# Test the optimized version
print("\n\nEvaluating Version 3 (Optimized)...")
results_v3 = evaluator.evaluate_prompt(prompt_v3_optimized, "v3_optimized")

print(f"\nv3_optimized Results:")
print(f"  Pass Rate: {results_v3['pass_rate']:.1%}")
print(f"  Improvement over v1: {(results_v3['pass_rate'] - results_v1['pass_rate']):.1%}")
print(f"  Improvement over v2: {(results_v3['pass_rate'] - results_v2['pass_rate']):.1%}")

# Visualize all three versions
fig, ax = plt.subplots(figsize=(10, 6))

versions = ['v1_simple', 'v2_improved', 'v3_optimized']
pass_rates = [
    results_v1['pass_rate'],
    results_v2['pass_rate'],
    results_v3['pass_rate']
]

colors = ['#d62728' if pr < 0.7 else '#ff7f0e' if pr < 0.85 else '#2ca02c' for pr in pass_rates]

bars = ax.bar(versions, pass_rates, color=colors, alpha=0.8)
ax.axhline(y=0.7, color='red', linestyle='--', label='70% threshold', alpha=0.5)
ax.axhline(y=0.85, color='orange', linestyle='--', label='85% target', alpha=0.5)

ax.set_ylabel('Pass Rate', fontsize=12)
ax.set_title('Prompt Version Performance', fontsize=14, fontweight='bold')
ax.set_ylim(0, 1)
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar, rate in zip(bars, pass_rates):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{rate:.1%}',
            ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Auto-graded checks
assert results_v3['pass_rate'] >= results_v2['pass_rate'], "v3 should improve on v2"
assert results_v3['pass_rate'] >= 0.7, "v3 should achieve at least 70% pass rate"

print("\n✓ Exercise 3 passed!")

## Exercise 4: Cost-Performance Tradeoff Analysis

**Task**: Analyze the tradeoff between performance and cost.

Sometimes a simpler, cheaper prompt is good enough. Calculate:
- Cost per test (tokens × price per token)
- Cost per successful result
- Performance improvement vs. cost increase

In [ ]:
# YOUR CODE HERE
# Implement cost analysis

# Pricing (example - update with actual model pricing)
PRICE_PER_1K_TOKENS = 0.008  # $0.008 per 1K tokens for Claude Sonnet

def calculate_cost_metrics(results: Dict) -> Dict:
    """
    Calculate cost-related metrics.
    
    Returns
    -------
    dict
        cost_per_test, cost_per_success, total_cost
    """
    # YOUR CODE HERE
    total_tokens = results['total_tokens']
    total_cost = (total_tokens / 1000) * PRICE_PER_1K_TOKENS
    cost_per_test = total_cost / results['total_tests']
    cost_per_success = total_cost / results['passed_tests'] if results['passed_tests'] > 0 else float('inf')
    
    return {
        'total_cost': total_cost,
        'cost_per_test': cost_per_test,
        'cost_per_success': cost_per_success
    }

# Calculate for all versions
print("Cost-Performance Analysis")
print("="*60)

all_results = [results_v1, results_v2, results_v3]
cost_data = []

for results in all_results:
    costs = calculate_cost_metrics(results)
    cost_data.append(costs)
    
    print(f"\n{results['version'].upper()}:")
    print(f"  Pass Rate: {results['pass_rate']:.1%}")
    print(f"  Total Cost: ${costs['total_cost']:.4f}")
    print(f"  Cost per Test: ${costs['cost_per_test']:.4f}")
    print(f"  Cost per Success: ${costs['cost_per_success']:.4f}")
    print(f"  Efficiency: {results['pass_rate'] / costs['cost_per_test']:.1f} success/cent")

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Pass rate vs cost per test
versions = [r['version'] for r in all_results]
pass_rates = [r['pass_rate'] for r in all_results]
costs_per_test = [cd['cost_per_test'] for cd in cost_data]

ax1.scatter(costs_per_test, pass_rates, s=200, alpha=0.6)
for i, v in enumerate(versions):
    ax1.annotate(v, (costs_per_test[i], pass_rates[i]), 
                xytext=(5, 5), textcoords='offset points')
ax1.set_xlabel('Cost per Test ($)', fontsize=11)
ax1.set_ylabel('Pass Rate', fontsize=11)
ax1.set_title('Cost vs Performance', fontsize=12, fontweight='bold')
ax1.grid(alpha=0.3)

# Cost per success comparison
costs_per_success = [cd['cost_per_success'] for cd in cost_data]
ax2.bar(versions, costs_per_success, alpha=0.8, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
ax2.set_ylabel('Cost per Success ($)', fontsize=11)
ax2.set_title('Cost Efficiency', fontsize=12, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Determine best value
best_value_idx = min(range(len(cost_data)), 
                     key=lambda i: cost_data[i]['cost_per_success'])
print(f"\nBest Value: {versions[best_value_idx]}")

# Auto-graded checks
assert all(cd['cost_per_test'] > 0 for cd in cost_data), "All versions should have positive cost"
print("\n✓ Exercise 4 passed!")

## Exercise 5: Create a Regression Test Suite

**Task**: Package your test suite for continuous testing.

Create a function that:
1. Loads test cases from a file
2. Runs evaluation on current prompt version
3. Compares against baseline results
4. Alerts if performance degrades

In [ ]:
# YOUR CODE HERE
# Implement regression testing framework

class RegressionTester:
    """
    Framework for ongoing prompt quality assurance.
    """
    
    def __init__(self, evaluator: PromptEvaluator, baseline_version: str):
        self.evaluator = evaluator
        self.baseline_version = baseline_version
        
        # Get baseline results
        baseline_results = [r for r in evaluator.results 
                          if r.prompt_version == baseline_version]
        
        # Calculate baseline metrics
        self.baseline_metrics = {}
        for metric in baseline_results[0].scores.keys():
            scores = [r.scores[metric] for r in baseline_results]
            self.baseline_metrics[metric] = {
                'mean': np.mean(scores),
                'min': np.mean(scores) - 2 * np.std(scores)  # 2 sigma threshold
            }
    
    def test_version(self, prompt_template: str, version_name: str, 
                    tolerance: float = 0.05) -> Dict:
        """
        Test a new prompt version against baseline.
        
        Parameters
        ----------
        prompt_template : str
            Prompt to test
        version_name : str
            Version identifier
        tolerance : float
            Acceptable performance degradation (e.g., 0.05 = 5%)
        
        Returns
        -------
        dict
            Test results with pass/fail and degradation analysis
        """
        # YOUR CODE HERE
        # 1. Run evaluation
        # 2. Compare to baseline
        # 3. Check if any metric dropped more than tolerance
        # 4. Return detailed report
        
        results = self.evaluator.evaluate_prompt(prompt_template, version_name)
        
        report = {
            'version': version_name,
            'regression_detected': False,
            'degraded_metrics': [],
            'metric_comparison': {}
        }
        
        for metric, baseline in self.baseline_metrics.items():
            current_mean = results['metrics'][metric]['mean']
            baseline_mean = baseline['mean']
            degradation = baseline_mean - current_mean
            
            report['metric_comparison'][metric] = {
                'baseline': baseline_mean,
                'current': current_mean,
                'degradation': degradation,
                'degradation_pct': degradation / baseline_mean if baseline_mean > 0 else 0
            }
            
            if degradation > tolerance:
                report['regression_detected'] = True
                report['degraded_metrics'].append(metric)
        
        return report

# Test the regression tester
regression_tester = RegressionTester(evaluator, "v3_optimized")

# Create a deliberately degraded prompt to test detection
degraded_prompt = """Analyze this: {text}
Give sentiment as JSON.
"""

report = regression_tester.test_version(degraded_prompt, "v4_degraded")

print("Regression Test Report")
print("="*60)
print(f"Version: {report['version']}")
print(f"Regression Detected: {report['regression_detected']}")

if report['degraded_metrics']:
    print(f"\nDegraded Metrics: {', '.join(report['degraded_metrics'])}")

print("\nMetric Comparison:")
for metric, comparison in report['metric_comparison'].items():
    print(f"  {metric}:")
    print(f"    Baseline: {comparison['baseline']:.3f}")
    print(f"    Current: {comparison['current']:.3f}")
    print(f"    Change: {comparison['degradation_pct']:.1%}")

# Auto-graded checks
assert report['regression_detected'], "Should detect regression in degraded prompt"
print("\n✓ Exercise 5 passed!")

## Summary

Congratulations! You've built a comprehensive prompt testing and evaluation system. Key takeaways:

1. **Systematic testing is essential** - Don't deploy prompts without validation
2. **Multiple metrics matter** - Track format, accuracy, completeness, cost
3. **Statistical rigor** - Use proper significance testing for A/B comparisons
4. **Cost-performance tradeoffs** - Sometimes simpler is better
5. **Regression testing** - Prevent degradation as prompts evolve

## Best Practices

- Maintain a diverse test suite covering edge cases
- Set clear acceptance thresholds (e.g., 85% pass rate)
- Track performance over time, not just point-in-time
- Consider cost alongside quality metrics
- Automate testing in CI/CD pipelines
- Document why specific prompts perform better

## Production Deployment Checklist

Before deploying a prompt to production:

- [ ] Pass rate ≥ 85% on test suite
- [ ] No regression vs. baseline
- [ ] Cost per request within budget
- [ ] Latency meets SLA requirements
- [ ] Edge cases handled gracefully
- [ ] Output format validated
- [ ] Monitoring and alerting configured

## Next Steps

- **Module 2**: Enhance prompts with RAG for factual grounding
- **Module 3**: Build custom applications with tested prompts
- **Module 4**: Deploy with monitoring and quality tracking